# **Analysis of Illegal Advertisements**
---

### Install packages

In [1]:
!pip install pandas openai google-generativeai faiss-cpu tiktoken
!pip install --upgrade pymilvus 
!pip install "pymilvus[model]"

  Using cached grpcio-1.71.0-cp313-cp313-macosx_10_14_universal2.whl.metadata (3.8 kB)
Using cached grpcio-1.71.0-cp313-cp313-macosx_10_14_universal2.whl (11.3 MB)
  Attempting uninstall: grpcio
    Found existing installation: grpcio 1.67.1
    Uninstalling grpcio-1.67.1:
      Successfully uninstalled grpcio-1.67.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pymilvus 2.5.10 requires grpcio<=1.67.1,>=1.49.1, but you have grpcio 1.71.0 which is incompatible.

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
  Using cached grpcio-1.67.1-cp313-cp313-macosx_10_13_universal2.whl.metadata (3.9 kB)
Using cached grpcio-1.67.1-cp313-cp313-macosx_10_13_universal2.whl (10.9 MB)
  Attempting uninstall: grpcio
    Found existing installation: grpcio 1.71.0
    Uninstalling grpcio-1.71.0:
      Successfully 

### Import packages

In [ ]:
import numpy as np
import pandas as pd
import os, faiss, textwrap, pickle, time, openai, tiktoken
import google.generativeai as genai
from tqdm import tqdm
from pymilvus import model

/Users/xiechengxian/Documents/人工智慧/Final_project/AI_final_project_env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Load data

In [3]:
DATA_PATH = "formatted_cases.txt"

### Embedding

In [ ]:
EMBEDDING_MODEL = "text-embedding-3-large"
OPENAI_API_KEY = "YOUR_OPENAI_API_KEY"
MAX_TOKENS = 8191

client = openai.OpenAI(api_key=OPENAI_API_KEY)
tokenizer = tiktoken.encoding_for_model(EMBEDDING_MODEL)

In [5]:
def chunk_text(text, max_tokens=MAX_TOKENS):
    tokens = tokenizer.encode(text)
    chunks = []
    for i in range(0, len(tokens), max_tokens):
        chunk = tokenizer.decode(tokens[i:i+max_tokens])
        chunks.append(chunk)
    return chunks

def embed_with_retry(texts, model=EMBEDDING_MODEL, retries=3, wait=10):
    for attempt in range(retries):
        try:
            response = client.embeddings.create(model=model, input=texts)
            return [np.array(e.embedding, dtype=np.float32) for e in response.data]
        except Exception as e:
            print(f"⚠️ error: {e}, retry {attempt + 1}/{retries}...")
            time.sleep(wait)
    raise RuntimeError("Failed to get embeddings after multiple attempts.")

def get_embeddings(text_list, model=EMBEDDING_MODEL):
    final_embeddings = []

    for text in text_list:
        chunks = chunk_text(text)
        
        if len(chunks) == 1:
            embeddings = embed_with_retry([chunks[0]], model=model)[0]
        else:
            chunk_embeddings = embed_with_retry(chunks, model=model)
            embeddings = np.mean(chunk_embeddings, axis=0)
        
        final_embeddings.append(embeddings)

    return final_embeddings

In [6]:
# Texts embedding
BATCH_SIZE = 50

with open(DATA_PATH, "r", encoding="utf-8") as f:
    text = f.read()

chunks = text.split("\n\n")  # Each chunk is separated by two newlines

texts = []
metadata = []
for chunk in chunks:
    if chunk.strip() == "":
        continue
    texts.append(chunk)
    metadata.append({"raw": chunk})  # Storing the raw text as metadata

texts_embeddings = []
for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="Embedding chunks"):
    batch = texts[i:i+BATCH_SIZE]
    batch_embeddings = get_embeddings(batch)
    texts_embeddings.extend(batch_embeddings)

# Store to FAISS
embedding_dim = len(texts_embeddings[0])
index = faiss.IndexFlatL2(embedding_dim)
index.add(np.vstack(texts_embeddings))

Embedding chunks: 100%|██████████| 11/11 [04:45<00:00, 25.99s/it]


In [8]:
# Query embeddings
QUERY_PATH = "final_project_query.csv"

df = pd.read_csv(QUERY_PATH)
query_data = list(enumerate(df.iloc[:, 1]))

query_embeddings = []
for idx, ad_text in tqdm(query_data, desc="Embedding queries"):
    vector = get_embeddings([ad_text])[0]
    query_embeddings.append((idx, vector))

query_embedding_dict = {id: vector for id, vector in query_embeddings}

Embedding queries: 100%|██████████| 100/100 [00:59<00:00,  1.67it/s]


In [9]:
# Save the index, metadata and query embeddings
os.makedirs("embedding_vectors", exist_ok=True)
INDEX_PATH = "embedding_vectors/openai_faiss_index.index"
METADATA_PATH = "embedding_vectors/openai_metadata.pkl"
QUERY_EMBEDDINGS_PATH = "embedding_vectors/openai_query_embeddings.pkl"

faiss.write_index(index, INDEX_PATH)

with open(METADATA_PATH, "wb") as f:
    pickle.dump(metadata, f)

with open(QUERY_EMBEDDINGS_PATH, "wb") as f:
    pickle.dump(query_embedding_dict, f)

### Retrieval

In [11]:
# Reload the index and metadata
INDEX_PATH = "embedding_vectors/openai_faiss_index.index"
METADATA_PATH = "embedding_vectors/openai_metadata.pkl"
QUERY_EMBEDDINGS_PATH = "embedding_vectors/openai_query_embeddings.pkl"

index = faiss.read_index(INDEX_PATH)

with open(METADATA_PATH, "rb") as f:
    metadata = pickle.load(f)
with open(QUERY_EMBEDDINGS_PATH, "rb") as f:
    query_embedding_dict = pickle.load(f)

In [19]:
def retrieve_similar_cases(query_embedding, k=3):
    distances, indices = index.search(query_embedding.reshape(1, -1), k)
    return [metadata[i]["raw"] for i in indices[0]]

In [20]:
# Test
query = "本產品可以幫助調整腸胃道機能"
similar_cases = retrieve_similar_cases(get_embeddings([query])[0])

for case in similar_cases:
    print(case, end="\n\n")

【主張】：幫助維持消化道機能
【判定】：此類詞句為食品及相關產品標示宣傳廣告中允許使用的詞句。
【原因/法條】：此詞句為一般性健康維持與生理機能輔助之合理宣稱，符合食品廣告應真實、客觀、不得誇大或作虛偽不實之表示的原則，且未涉及醫療效能或不實誇張之虞。

【主張】：幫助消化
【判定】：此類詞句為食品及相關產品標示宣傳廣告中允許使用的詞句。
【原因/法條】：此詞句為一般性健康維持與生理機能輔助之合理宣稱，符合食品廣告應真實、客觀、不得誇大或作虛偽不實之表示的原則，且未涉及醫療效能或不實誇張之虞。

【主張】：涉及保護消化道全機能的宣稱。
【判定】：此類詞句在食品、健康食品或一般產品廣告中，被認定為可使用的描述性詞句。
【原因/法條】：此詞句強調產品有助於維護消化系統的整體健康功能，屬於一般性健康維持與輔助，而非針對特定消化道疾病的治療或預防。其表述方式未有誇大不實或誤導消費者之虞，符合食品、藥品、健康食品廣告相關法規中對於非醫療效能宣傳的原則。



### Querying

In [ ]:
GEMINI_API_KEY = "YOUR_GEMINI_API_KEY"
LLM_MODEL = "gemini-2.5-flash-preview-05-20"

genai.configure(api_key=GEMINI_API_KEY)
llm_model = genai.GenerativeModel(LLM_MODEL)

In [15]:
system_prompt = textwrap.dedent("""\
你是一位廣告審查專家，請根據以下法規來判斷一則廣告內容是否合法，並解釋相關法條內容及原因。
                            
### 合規性判斷
- **無罪判定原則**：不捏造或過度解讀廣告文字，**從寬認定合法性**，但如果是**藥物**宣稱**科學實證**、**國外研究**一類用語，則提高違法可能性認定，除非內容中出現完整的**衛福部核可字號**或**衛福部認證**。
- **比對允許使用的廣告用詞**：
- 「完整補充營養」「調整體質」「促進新陳代謝」「幫助入睡」「保護消化道全機能」「改變細菌叢生態」「排便有感」「在嚴謹的營養均衡與熱量控制，以及適當的運動條件下，適量攝取本產品有助於不易形成體脂肪」這些文字出現時不視為有違法風險。
- 「能完整補充人體營養」、「調整體質」、「提升生理機能」、「調節生理機能」、「促進新陳代謝」、「幫助入睡」、「調整體質」、「青春美麗」、「排便超有感」、「給你排便順暢新體驗」、「維持正常的排便習慣」、「排便順暢」、「促進新陳代謝」、「調整體質」、「改變細菌叢生態」、調節生理機能」、「保護消化道全機能」、「提升吸收滋養消化機能」"這些文字出現時不視為有違法風險。
                            
## 分析步驟
1. **解析廣告內容**：檢視是否涉及療效誇大。
2. **案例文件與法規比對**：根據以下法律文件與案例，提供比對結果。
3. **判斷違法機率**：依據法律及案例進行風險評估。
4. **裁罰依據**：
- ** 例：違反《食品安全衛生管理法》第28條第1項**
    
                            
## 法律文件
- 健康食品管理法  
    - 第14條: 健康食品之標示或廣告，不得涉及醫療效能之內容。
                            
- 食品安全衛生管理法
    - 第3條: 食品：指供人食用或飲用之產品及其原料。（定義條款，無罰則）
    - 第28條-第1項: 食品、食品添加物、食品用洗滌劑及經中央主管機關公告之食品器具、食品容器或包裝，其標示、宣傳或廣告不得涉及醫療效能。                   

- 化粧品衛生管理法
    - 第10條: 化粧品之標示、宣傳及廣告內容，不得有虛偽或誇大之情事。
                            
- 藥事法
    - 第66條-第1項: 藥商刊播藥物廣告時，應於刊播前將所有文字、圖畫或言詞，申請中央或直轄市衛生主管機關核准，未經核准或經核准後廢止者，不得刊播。
    - 第66條-第2項: 藥物廣告在核准登記、刊播期間不得變更原核准事項。
    - 第66條-第3項: 傳播業者不得刊播未經中央或直轄市衛生主管機關核准、與核准事項不符、已廢止或經令立即停止刊播並限期改善而尚未改善之藥物廣告。
    - 第66條-第4項: 受委託刊播之傳播業者，應自廣告之日起六個月，保存委託刊播廣告者之姓名（公司或團體名稱）、身分證或事業登記證字號、住居所（事務所或營業所）及電話等資料，且於主管機關要求提供時，不得規避、妨礙或拒絕。
    - 第68條: 藥物廣告不得以左列方式為之：一、假借他人名義為宣傳者。二、利用書刊資料保證其效能或性能。三、藉採訪或報導為宣傳。四、以其他不正當方式為宣傳。
    - 第69條: 非本法所稱之藥物，不得為醫療效能之標示或宣傳。
                            
- 醫療器材管理法
    - 第6條: 醫療器材廣告指利用傳播方法，宣傳醫療效能，以達招徠銷售醫療器材為目的之行為。採訪、報導或宣傳之內容暗示或影射醫療器材之醫療效能，以達招徠銷售醫療器材為目的者，視為醫療器材廣告。
    - 第40條: 非醫療器材商不得為醫療器材廣告。
    - 第41條-第1項: 醫療器材商刊播醫療器材廣告時，應由許可證所有人或登錄者於刊播前，檢具廣告所有文字、圖畫或言詞，依醫療器材商登記所在地，在直轄市者向直轄市主管機關，在縣（市）者向中央主管機關，申請核准刊播；經核准後，應向傳播業者送驗核准文件，始得刊播。
    - 第41條-第2項: 醫療器材廣告於核准刊播期間，不得變更原核准事項而為刊播。
    - 第46條: 非醫療器材，不得為醫療效能之標示或宣傳。但其他法律另有規定者，不在此限。
                            
""")

In [29]:
def check_legality(ad_text, ad_embedding):
    full_prompt = textwrap.dedent(f"""{system_prompt}
    請分析以下廣告內容是否合法？

    ##【廣告句】
    {ad_text}

    ##【與廣告內容相關之參考案例】
    {textwrap.indent('\n\n'.join(retrieve_similar_cases(ad_embedding)), '    ')}

    請根據上述法律與案例，判斷廣告內容是否合法，並提供相關法條內容及解釋。  
        
    輸出格式：
    <0(不合法)或1(合法)>

    相關法條內容及解釋:
    <說明>                                           
    """)
    response = llm_model.generate_content(full_prompt)
    return response.text.strip()

def process_row(ad_text, ad_embedding):
    result = check_legality(ad_text, ad_embedding)
    lines = result.strip().splitlines()
    if lines:
        answer = lines[0].strip()
        answer = '1' if '1' in answer else '0'
    else:
        answer = '0'

    return {"ID": idx, "Answer": answer, "FullResponse": result}

MAX_RETRIES = 5
RETRY_DELAY = 5
def robust_process_row(ad_text, ad_embedding):
    retries = 0
    while retries < MAX_RETRIES:
        try:
            return process_row(ad_text, ad_embedding)
        except Exception as e:
            retries += 1
            print(f"[ID {idx}] DeadlineExceeded, retrying {retries}/{MAX_RETRIES} in {RETRY_DELAY} seconds...")
            time.sleep(RETRY_DELAY)

    print(f"[ID {idx}] ❌ Failed after {MAX_RETRIES} retries.")
    return {
        "ID": idx,
        "Answer": "Timeout",
        "FullResponse": "DeadlineExceeded after max retries"
    }

In [33]:
# Test
example_ad = "酵素 - Google Shopping 三多消化酵素Plus膜衣錠60錠Visit siteNT$446.00 · 三多sentosa+NT$110.00 shippingCOMPARE PRICESPRODUCT DETAILS添加更多種不同功能性的酵素、含澱粉分解酵素、脂肪分解酵素、蛋白質分解酵素、鳳梨酵素、木瓜酵素,幫助食物中三大營養素(醣類、脂肪、蛋白質)分解與消化,有助於營養素吸收。 B群加強能量代謝:含維生素B1、B2、B6、泛酸,促進新陳代謝。 特別添加芽孢乳酸菌:可順利通過消化道,可幫助維持消化道機能、改變細菌叢生態、健康維持。 添加鎂,有助於身體正常代謝。 幫助消化 促進新陳代謝 品 名:三多消化酵素Plus膜衣錠 原料成分:微結晶狀α-纖維素、綜合酵素(麥芽糊精、澱粉分解酵素、植酸酵素、葡萄糖澱粉酵素、複合酵素(脂肪分解酵素、蛋白質分解酵素、澱粉分解酵素、麥芽糊精)、中性蛋白質分解酵素、鳳梨酵素、蛋白質分解酵素4.5、轉化酵素、蛋白質分解酵素3.0、半纖維素分解酵素、果膠酵素、α-半乳糖苷酶、蛋白質分解酵素、脂肪分解酵素、胜肽分解酵素、乳糖分解酵素、蛋白質分解酵素6.0、澱粉酵素、纖維素分解酵素)、綜合酵素Plus(麥芽糊精、澱粉分解酵素、蛋白質分解酵素、脂肪分解酵素、纖維分解酵素)、啤酒酵母(啤酒酵母、麥芽萃取)氧化鎂、青木瓜萃取物(麥芽糊精、乳糖、青木瓜萃取物、益生菌發酵物粉末(唾液乳桿菌發酵粉、乳雙岐桿菌發酵粉、長雙岐桿菌發酵粉、麥芽糊精、檸檬酸)羥丙基甲基纖維素、本多酸鈣、二氧化鈦、食用黃色四號鋁麗基、乳酸菌(Bacillus Coagulans、乳糖)維生素B1、維生素B2、維生素B6、食用黃色五號鋁麗基、食用藍色一號鋁麗基。 (食品添加物符合國家安全標準,請安心食用) 容量:0.5公克/錠,60錠/盒 保存期限:三年 有效日期:日/月/西元年 貨源: 公司貨 用法用量: 1、每餐飯後1錠。 2、請搭配溫冷開水食用。"
print(check_legality(example_ad, get_embeddings([example_ad])[0]))

ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 500
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, retry_delay {
  seconds: 54
}
]

In [32]:
# Test
similar_cases = retrieve_similar_cases(query_embedding_dict.get(1))
for case in similar_cases:
    print(case, end="\n\n")

【主張】：受到經前症候群的困擾..包括有焦慮、緊張、倦怠、乳房腫脹..服用OO酵素後都有明顯的改善
【判定】：此類詞句（整體表現「涉及生理功能者」）被認定為食品廣告中誇大、易生誤解，且涉及醫療效能，為不適當且違規的廣告詞句。
【原因/法條】：依據《食品安全衛生管理法》等相關法規，食品廣告不得宣稱醫療效能或為誇大、易生誤解之表示。此類詞句直接聲稱產品能「明顯改善」經前症候群的各種症狀（焦慮、緊張、倦怠、乳房腫脹），涉及對疾病症狀的治療或改善，超出食品的合理範疇。

【主張】：幫助腸內有益菌的增生
【判定】：超出胃腸功能改善之保健功效許可範圍
【原因/法條】：健康食品的「胃腸功能改善」保健功效通常指有助於增加腸內益生菌數量或改善排便順暢。宣稱「幫助腸內有益菌的增生」已超出此核准範圍，可能涉及誇大或暗示特定生理作用，違反《健康食品管理法》第14條及第15條規定。

【主張】：促進乳酸菌復活、活化腸道細菌
【判定】：超出胃腸功能改善之保健功效許可範圍
【原因/法條】：健康食品的「胃腸功能改善」保健功效通常指有助於增加腸內益生菌數量或改善排便順暢。宣稱「促進乳酸菌復活」、「活化腸道細菌」已超出此核准範圍，暗示對特定細菌的「活化」作用，可能涉及誇大或醫療效能，違反《健康食品管理法》第14條及第15條規定。



In [31]:
os.makedirs("results", exist_ok=True)
QUERY_PATH = "final_project_query.csv"
OUTPUT_PATH = "results/final_project_results.csv"

df = pd.read_csv(QUERY_PATH)
input_data = list(enumerate(df.iloc[:, 1]))

results = []
for idx, ad_text in tqdm(input_data, desc="processing ads"):
    results.append(robust_process_row(ad_text, query_embedding_dict.get(idx)))

df_result = pd.DataFrame([{"ID": r["ID"], "Answer": r["Answer"]} for r in results])
df_result.to_csv(OUTPUT_PATH, index=False)

processing ads:   0%|          | 0/100 [00:00<?, ?it/s]

[ID 0] DeadlineExceeded, retrying 1/5 in 5 seconds...
[ID 0] DeadlineExceeded, retrying 2/5 in 5 seconds...


processing ads:   0%|          | 0/100 [00:08<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
# Write full responses to a markdown file
RESPONSE_LOG_PATH = "results/final_project_raw_responses.md"

with open(RESPONSE_LOG_PATH, "w", encoding="utf-8") as f:
    for r in results:
        f.write(f"--- ID {r['ID']} ---\n")
        f.write(r["FullResponse"] + "\n\n")